In [3]:
import multiprocessing
import time
import random

def worker(task_queue, results_queue):
    while True:
        task = task_queue.get()
        if task is None:  # The "Poison Pill"
            print(f"Worker {multiprocessing.current_process().name} shutting down.")
            break
        
        # Simulate heavy CPU work
        try:
            result = f"Processed {task} by {multiprocessing.current_process().name}"
            time.sleep(random.uniform(0.1, 0.5)) 
            results_queue.put(result)
        except Exception as e:
            results_queue.put(f"ERROR on {task}: {e}")

if __name__ == "__main__":
    tasks = multiprocessing.Queue()
    results = multiprocessing.Queue()
    num_workers = multiprocessing.cpu_count()
    
    # 1. Start Workers
    processes = [multiprocessing.Process(target=worker, args=(tasks, results)) for _ in range(num_workers)]
    for p in processes: p.start()

    # 2. Producer: Feed the Queue
    for i in range(20):
        tasks.put(f"Job_{i}")

    # 3. Feed the Poison Pills (One for each worker)
    for _ in range(num_workers):
        tasks.put(None)

    for p in processes: p.join()
    
    # 4. Collect Results
    while not results.empty():
        print(results.get())

In [1]:
import multiprocessing

def update_shared_stats(worker_id, shared_dict, lock):
    # Simulating work and updating a shared "Dashboard"
    for i in range(100):
        with lock: # Critical: Prevent Race Conditions on the shared dict
            shared_dict['total_iterations'] += 1
            shared_dict['last_worker'] = worker_id

if __name__ == "__main__":
    with multiprocessing.Manager() as manager:
        # This dict exists in a server process and is synced across all forks
        dashboard = manager.dict({'total_iterations': 0, 'last_worker': None})
        lock = manager.Lock()
        
        pool = []
        for i in range(4):
            p = multiprocessing.Process(target=update_shared_stats, args=(i, dashboard, lock))
            pool.append(p)
            p.start()

        for p in pool: p.join()

        print(f"Final Dashboard State: {dashboard}")

Final Dashboard State: {'total_iterations': 0, 'last_worker': None}


In [2]:
import multiprocessing
import ctypes
import numpy as np

def compute_on_shared(idx, shape):
    # Access existing memory buffer without copying
    existing_shmem = np.frombuffer(shared_array_base.get_obj(), dtype=ctypes.c_double)
    arr = existing_shmem.reshape(shape)
    arr[idx] = np.sin(idx) # Write directly to shared memory

if __name__ == "__main__":
    size = 1000000
    # Allocate raw shared memory
    shared_array_base = multiprocessing.Array(ctypes.c_double, size)
    # Process...

ModuleNotFoundError: No module named 'numpy'

In [3]:
def stage_worker(barrier, worker_id):
    print(f"Worker {worker_id} finished Stage 1")
    barrier.wait() # All processes pause here until the count hits 4
    print(f"Worker {worker_id} starting Stage 2")
    

In [4]:
p = multiprocessing.Process(target=heavy_task)
p.start()
p.join(timeout=10) 
if p.is_alive():
    print("Task exceeded 10s. Terminating...")
    p.terminate() # Force kill

NameError: name 'heavy_task' is not defined